### Document Loader Using PyPDFLoader

In [1]:
from langchain_classic.document_loaders import PyPDFLoader

d:\Personal\MineProjects\PDF_RAG_ChatBot\project\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import os
filepath = os.path.join("Dummy File.pdf")
loader = PyPDFLoader(filepath)

docs = loader.load()
print(docs[0].page_content,end=" ")

A large language model (LLM) is a neural network trained on a vast amount of text for natural 
language processing tasks, especially language generation. LLMs can typically generate, 
summarize, translate, and analyze text in many contexts, and are a foundational technology 
behind modern chatbots.[1] Biased or inaccurate training data can make an LLM's output less 
reliable.[2] 
 
LLMs are typically based on transformer architecture, which is more parallelizable than earlier 
recurrent neural network models.[3] Generative pre-trained transformers (GPTs) are a type of 
LLM that is pre-trained to predict the next word.[4] GPTs are then often fine-tuned to follow 
instructions and to behave as assistants.[5] 
 
Benchmark evaluations for LLMs attempt to measure model reasoning, factual accuracy, 
alignment, and safety.[6] 

In [3]:
from langchain_classic.text_splitter import RecursiveCharacterTextSplitter

In [26]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=250,
    chunk_overlap=50,
)
splitted_doc = splitter.split_documents(docs)
print(splitted_doc[2].page_content)

reliable.[2] 
 
LLMs are typically based on transformer architecture, which is more parallelizable than earlier 
recurrent neural network models.[3] Generative pre-trained transformers (GPTs) are a type of


In [5]:
from langchain_huggingface import HuggingFaceEmbeddings
# Initialize the selected class
embedding_model = HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6490.68it/s]


In [27]:
split_contents = [doc.page_content for doc in splitted_doc]
doc_embed = embedding_model.embed_documents(split_contents)
print(doc_embed)

# here we done embedding manually but it is handled by vector db qdrant

[[0.04479087516665459, -0.1043391302227974, 0.016437016427516937, -0.005880993790924549, 0.04688015580177307, 0.030024027451872826, -0.06483811885118484, -0.004736484959721565, 0.04714062809944153, -0.002993781352415681, -0.019733600318431854, 0.006883115507662296, 0.06764670461416245, -0.03187219053506851, -0.015572020784020424, 0.05920944735407829, 0.08726504445075989, 0.005941744428128004, -0.030568474903702736, -0.080954909324646, 0.036293499171733856, 0.07900267094373703, 0.023512715473771095, 0.00626571848988533, 0.00720028905197978, 0.02696901001036167, -0.0017929113237187266, 0.031005674973130226, 0.051012445241212845, -0.015527593903243542, 0.03082035854458809, 0.07319462299346924, 0.028574975207448006, 0.07221782952547073, -0.07606785744428635, 0.05740773305296898, -0.10201055556535721, -0.03753483295440674, 0.007605019491165876, -0.026552071794867516, 0.0025579784996807575, -0.008320454508066177, 0.058542683720588684, -0.031675904989242554, 0.13532806932926178, 0.01889508031

In [7]:
from langchain_qdrant import QdrantVectorStore

In [28]:
vector_store = QdrantVectorStore.from_documents(
    documents=splitted_doc,
    embedding=embedding_model,
    location=":memory:", 
    collection_name="pdf_chat_history",
)

In [49]:
ques = "What is MS Dhoni"
retriever = vector_store.as_retriever(search_kwargs={"k": 2})
retrieved_chunks = retriever.invoke(query)

context = "\n\n".join([doc.page_content for doc in retrieved_chunks])

In [30]:
from langchain_classic.prompts import PromptTemplate

from langchain_groq import ChatGroq

In [47]:
SystemPrompt = PromptTemplate(
    template="""
    You are helpful AI Assistant and your goal is to answer the relevant query of user from given retrieved context only.
    Here are given User Query & Context Retrieved
    User Query : {query}
    Context : {context}

    Note : If user query is out of syllabus from given Context Just Say "Don't have Relevant information about your query" and paste user query after this line.
    """,
    input_variables=["query","context"]
)

In [37]:
from dotenv import load_dotenv
load_dotenv()

True

In [44]:
from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()

In [50]:
model = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0.2, # Lower temperature makes the AI stick strictly to the PDF facts
)

chain = SystemPrompt | model | parser

result = chain.invoke({
    "query" : ques,
    "context" : context
})

print(result)

Don't have Relevant information about your query
What is MS Dhoni
